# CO3133: Deep Learning and Its Applications (Semester 261)
## Assignment 01: Foundations of Deep Learning Pipelines and Architectures
### Comparative Study: From Linear Models to Modern Sequence Models for Image Classification
**Group:** Group Goat  
**Sole Contributor:** **Hoàng Trọng Huy Minh** (Student ID: **2452743**) — **100% Contribution**  
*(Võ Lê Thành Triệu: 0%, Nguyễn Huỳnh Nam Quốc: 0%)*  
**Institution:** Faculty of Computer Science and Engineering, HCMUT - VNU-HCM  
**Instructor:** Assoc. Prof. Lê Thành Sách  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/volethanhtrieu/DL_Assignment-01/blob/main/assignment_01_reproducibility.ipynb)

---
### 🔗 Official Project Resources & Deliverables
- **Live GitHub Pages Webpage:** [Assignment 01 Webpage](https://volethanhtrieu.github.io/Deep-Learning-project-main-page/assignment-01/)
- **Source Code Repository:** [DL_Assignment-01 GitHub](https://github.com/volethanhtrieu/DL_Assignment-01)
- **Cumulative Technical Report (20 Pages):** `Goat_A1_Report.pdf`
- **Oral Defense Presentation Slides (PDF):** `docs/Slides_Assignment_01.pdf`
- **Automated Sanity Audit Suite:** `audit_sanity.py`

---
### 📋 Notebook Overview & Reproduction Objectives
This interactive Jupyter Notebook demonstrates the entire lifecycle of Assignment 1, enabling 100% independent replication:
1. **Google Colab & Environment Setup:** Automatic cloning and package installation if executed in Colab; seamless local execution.
2. **Data Pipeline & Leakage-Free Splitting:** Stratified 50k / 10k / 10k partition on Fashion-MNIST with zero sample overlap.
3. **Exploratory Data Analysis (EDA):** Class distribution, sample grids, class prototypes, and pixel intensity distributions.
4. **The 5 Architecture Families:** PyTorch implementations of Linear Classifier, MLP, Custom 3-Stage CNN, Bidirectional LSTM, and Vision Transformer (ViT).
5. **Unified Optimization Harness:** AdamW, Cosine Annealing, and Early Stopping on validation Macro-F1.
6. **Master Comparative Benchmark:** Evaluating all 5 pre-trained checkpoints on the 10,000 unseen test set.
7. **Qualitative Error Taxonomy:** Confusion matrices, per-class F1 progression, and failure case visualisations.
8. **Automated Sanity Audits:** In-line execution of all 4 integrity checks (leakage, gradient flow, determinism, checkpoints).


---
## Section 1: Automatic Environment Setup (Google Colab / Local)

If this notebook is opened in **Google Colab**, the cell below automatically clones the official GitHub repository, changes the working directory, and installs dependencies. If running locally, it detects your local workspace seamlessly.


In [ ]:
import os
import sys

# Detect if executing inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print(">>> Detected Google Colab environment.")
    print(">>> Cloning official repository to access checkpoints and code...")
    if not os.path.exists('DL_Assignment-01'):
        !git clone https://github.com/volethanhtrieu/DL_Assignment-01.git
    %cd DL_Assignment-01
    print(">>> Installing dependencies from requirements.txt...")
    !pip install -q -r requirements.txt
    print(">>> Google Colab workspace initialized successfully!\n")
else:
    print(">>> Running in local workspace. Environment verified.\n")

# Ensure root directory is in Python module search path
if '.' not in sys.path:
    sys.path.append('.')


---
## Section 2: Deterministic Seeding & Hardware Verification

We enforce strict deterministic execution across PyTorch, NumPy, and Python's random library using `seed=42`, and detect hardware acceleration (CUDA on NVIDIA GPU / Colab GPU).


In [ ]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch Version: {torch.__version__}")
print(f"Device Selected: {device}")
if device.type == 'cuda':
    print(f"GPU Hardware:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("GPU not detected; running on CPU.")


---
## Section 3: Data Pipeline, Normalization & Leakage-Free Stratified Splitting

Per Course Handbook Section 10 and Section 3.1:
- **Primary Dataset:** Fashion-MNIST (10 apparel categories, 28x28 grayscale).
- **Stratified Partition:**
  - **Training Set:** 50,000 instances (exactly 5,000 per class).
  - **Validation Set:** 10,000 instances (exactly 1,000 per class).
  - **Test Set:** 10,000 instances (official unseen benchmark partition).
- **Zero Leakage:** $\mathcal{S}_{\text{train}} \cap \mathcal{S}_{\text{val}} = \emptyset$.
- **Normalization:** $\mu = 0.2860, \sigma = 0.3530$.


In [ ]:
from data_setup import get_fashion_mnist_dataloaders

train_loader, val_loader, test_loader, class_names = get_fashion_mnist_dataloaders(
    data_dir="./data",
    batch_size=64,
    num_workers=0,
    pin_memory=True if device.type == 'cuda' else False,
    seed=42
)

train_set = train_loader.dataset
val_set = val_loader.dataset
test_set = test_loader.dataset

print(f"Dataset Split Sizes:")
print(f"  - Train Partition: {len(train_set):,} samples ({len(train_set)//10:,} per class)")
print(f"  - Val Partition:   {len(val_set):,} samples ({len(val_set)//10:,} per class)")
print(f"  - Test Partition:  {len(test_set):,} samples ({len(test_set)//10:,} per class)")

# Mathematical verification of zero data leakage
train_idx = set(train_set.indices)
val_idx = set(val_set.indices)
overlap = train_idx.intersection(val_idx)

print(f"\nData Leakage Check:")
print(f"  - Number of Overlapping Samples: {len(overlap)}")
assert len(overlap) == 0, "CRITICAL ERROR: Data leakage detected between train and val!"
print("  -> PASS: Strict instance-level disjointness verified.")


---
## Section 4: Exploratory Data Analysis (EDA)

We explore the 10 apparel classes:
`['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']`.
We visualize representative samples, compute mean class prototypes, and inspect the pixel intensity distribution.


In [ ]:
# 1. Visualise representative samples from each of the 10 apparel classes
plt.figure(figsize=(14, 5))
samples_per_class = {}

for images, labels in train_loader:
    for img, lbl in zip(images, labels):
        c = lbl.item()
        if c not in samples_per_class:
            # Denormalize image for display: img * std + mean
            raw_img = img.squeeze().numpy() * 0.3530 + 0.2860
            samples_per_class[c] = raw_img
        if len(samples_per_class) == 10:
            break
    if len(samples_per_class) == 10:
        break

for c in range(10):
    plt.subplot(2, 5, c + 1)
    plt.imshow(samples_per_class[c], cmap='gray', vmin=0, vmax=1)
    plt.title(f"[{c}] {class_names[c]}", fontsize=10, fontweight='bold')
    plt.axis('off')

plt.suptitle("Fashion-MNIST: Representative Instances Across 10 Categories", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 2. Compute and plot Mean Class Prototypes
print("Computing Mean Class Prototypes across 50,000 training samples...")
prototypes = {c: [] for c in range(10)}

for images, labels in train_loader:
    for img, lbl in zip(images, labels):
        raw_img = img.squeeze().numpy() * 0.3530 + 0.2860
        prototypes[lbl.item()].append(raw_img)

plt.figure(figsize=(14, 3.2))
for c in range(10):
    mean_img = np.mean(prototypes[c], axis=0)
    plt.subplot(1, 10, c + 1)
    plt.imshow(mean_img, cmap='magma', vmin=0, vmax=1)
    plt.title(f"{class_names[c]}", fontsize=8, fontweight='bold')
    plt.axis('off')

plt.suptitle("Class Prototypes (Mean Pixel Intensity Heatmaps)", fontsize=12, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()


---
## Section 5: The 5 Mandatory Architecture Families

We inspect and instantiate all 5 model families required by Handbook Section 11.1:
1. **Linear Classifier (Softmax Regression):** Flat 784 $\to$ 10 logits. CrossEntropyLoss on raw logits (**no premature softmax applied**).
2. **Multilayer Perceptron (MLP):** 784 $\to$ 256 $\to$ 128 $\to$ 10, with BatchNorm1d, ReLU, and Dropout(0.2).
3. **Custom 3-Stage CNN:** Conv2d ($3 \times 3$), BatchNorm, ReLU, MaxPool, AdaptiveAvgPool, and Linear classifier.
4. **Bidirectional LSTM:** Sequential row-by-row scanning ($28 \times 28$, $H=64$), 2 layers, concatenating forward and backward states $\to 128 \to 10$.
5. **Vision Transformer (ViT):** $4 \times 4$ patches (49 tokens + `[CLS]`), $D=48$, 3 encoder blocks, 4 heads, learnable 1D positional encodings.


In [ ]:
from models import get_model, count_parameters

models_dict = {
    "linear": get_model("linear"),
    "mlp": get_model("mlp"),
    "cnn": get_model("cnn"),
    "lstm": get_model("lstm"),
    "vit": get_model("vit")
}

param_summary = []
for name, m in models_dict.items():
    total_p, trainable_p = count_parameters(m, detailed=True)
    param_summary.append({
        "Model Name": name.upper(),
        "Architecture Family": m.__class__.__name__,
        "Trainable Parameters": f"{trainable_p:,}",
        "Total Parameters": f"{total_p:,}"
    })

df_params = pd.DataFrame(param_summary)
if 'display' in globals():
    display(df_params)
else:
    print(df_params.to_string(index=False))


---
## Section 6: Checkpoint Restoration & Strict Weight Verification

All five models were trained locally for 10 epochs using AdamW (lr=$10^{-3}$, weight decay=$10^{-4}$), Cosine Annealing, and Early Stopping on validation Macro-F1 (patience=4).  
We now restore the serialized checkpoints from `./outputs/checkpoints/` using `strict=True` to ensure complete parameter restoration.


In [ ]:
ckpt_dir = "./outputs/checkpoints"
loaded_models = {}

for name in ["linear", "mlp", "cnn", "lstm", "vit"]:
    ckpt_path = os.path.join(ckpt_dir, f"best_{name}.pth")
    assert os.path.exists(ckpt_path), f"Checkpoint missing: {ckpt_path}"
    
    ckpt = torch.load(ckpt_path, map_location=device)
    weights = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    
    model = get_model(name).to(device)
    model.load_state_dict(weights, strict=True)
    model.eval()
    loaded_models[name] = model
    
    size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
    epoch_info = ckpt.get("epoch", "N/A") if isinstance(ckpt, dict) else "N/A"
    f1_info = ckpt.get("val_macro_f1", "N/A") if isinstance(ckpt, dict) else "N/A"
    print(f"Successfully loaded {name.upper():<7} | Size: {size_mb:.2f} MB | Epoch: {epoch_info} | Best Val F1: {f1_info:.2f}% (Strict Load: PASS)")


---
## Section 7: Master 5-Model Comparative Benchmark

We evaluate all 5 models under strictly identical test conditions:
- **Test Set:** 10,000 samples, batch size 64.
- **Metrics:** Test Loss, Accuracy, Macro-F1, Trainable Parameter count, and Per-sample Inference Latency (ms).


In [ ]:
criterion = nn.CrossEntropyLoss()
results_list = []
all_predictions = {}
all_ground_truth = None

for name, model in loaded_models.items():
    total_loss = 0.0
    preds_list = []
    targets_list = []
    
    # Warm-up pass for precise timing
    with torch.no_grad():
        for i, (x_w, _) in enumerate(test_loader):
            if i >= 2: break
            _ = model(x_w.to(device))
    if device.type == 'cuda':
        torch.cuda.synchronize()
        
    t_start = time.perf_counter()
    with torch.no_grad():
        for x_b, y_b in test_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            out = model(x_b)
            loss = criterion(out, y_b)
            total_loss += loss.item() * len(y_b)
            preds_list.extend(out.argmax(dim=-1).cpu().numpy())
            targets_list.extend(y_b.cpu().numpy())
            
    if device.type == 'cuda':
        torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - t_start) * 1000 / len(targets_list)
    
    acc = accuracy_score(targets_list, preds_list) * 100
    macro_f1 = f1_score(targets_list, preds_list, average='macro') * 100
    avg_loss = total_loss / len(targets_list)
    params = count_parameters(model)
    
    all_predictions[name] = np.array(preds_list)
    if all_ground_truth is None:
        all_ground_truth = np.array(targets_list)
        
    results_list.append({
        "Model Architecture": name.upper(),
        "Milestone": "M1 Baseline" if name in ["linear", "mlp"] else "M2 Modern",
        "Trainable Params": f"{params:,}",
        "Test Loss": f"{avg_loss:.4f}",
        "Accuracy (%)": f"{acc:.2f}%",
        "Macro-F1 (%)": f"{macro_f1:.2f}%",
        "Inference Latency (ms)": f"{latency_ms:.3f} ms"
    })

df_results = pd.DataFrame(results_list)
if 'display' in globals():
    display(df_results)
else:
    print(df_results.to_string(index=False))


---
## Section 8: Per-Class F1 Score Progression Across Architectures

We analyze the fine-grained performance on each individual class to observe how inductive biases impact apparel recognition.


In [ ]:
per_class_data = []

for c in range(10):
    row = {"Class Index": c, "Class Name": class_names[c]}
    for name in ["linear", "mlp", "cnn", "lstm", "vit"]:
        f1_c = f1_score(all_ground_truth == c, all_predictions[name] == c, pos_label=True) * 100
        row[name.upper()] = f"{f1_c:.2f}%"
    per_class_data.append(row)

df_per_class = pd.DataFrame(per_class_data)
if 'display' in globals():
    display(df_per_class)
else:
    print(df_per_class.to_string(index=False))


---
## Section 9: Confusion Matrix & Qualitative Error Analysis

We compute and plot the normalized confusion matrix for the peak-performing Custom CNN model.


In [ ]:
cm_cnn = confusion_matrix(all_ground_truth, all_predictions["cnn"])
cm_norm = cm_cnn.astype('float') / cm_cnn.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Normalized Confusion Matrix — Custom 3-Stage CNN (M2 Final)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Predicted Class", fontsize=11, fontweight='bold')
plt.ylabel("Ground Truth Class", fontsize=11, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Print Top Failure Pairs
misclass_pairs = []
for i in range(10):
    for j in range(10):
        if i != j:
            misclass_pairs.append((cm_cnn[i, j], class_names[i], class_names[j]))
misclass_pairs.sort(reverse=True)

print("Top 5 Error Modes for Custom CNN:")
for count, true_cls, pred_cls in misclass_pairs[:5]:
    print(f"  - {true_cls:<12} misclassified as {pred_cls:<12} : {count} errors")


---
## Section 10: Qualitative Prediction Visualizations (Correct vs. Failure Cases)

We inspect test instances to identify what makes certain images easy or difficult for deep neural networks.


In [ ]:
model_cnn = loaded_models["cnn"]
model_cnn.eval()

correct_examples = []
failure_examples = []

with torch.no_grad():
    for x_b, y_b in test_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        logits = model_cnn(x_b)
        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        
        for i in range(len(y_b)):
            img = x_b[i].cpu().squeeze().numpy() * 0.3530 + 0.2860
            true_l = y_b[i].item()
            pred_l = preds[i].item()
            conf = probs[i, pred_l].item() * 100
            
            if true_l == pred_l and len(correct_examples) < 5:
                correct_examples.append((img, true_l, pred_l, conf))
            elif true_l != pred_l and len(failure_examples) < 5:
                failure_examples.append((img, true_l, pred_l, conf))
                
        if len(correct_examples) >= 5 and len(failure_examples) >= 5:
            break

plt.figure(figsize=(15, 6))
# Plot 5 Correct
for idx, (img, true_l, pred_l, conf) in enumerate(correct_examples):
    plt.subplot(2, 5, idx + 1)
    plt.imshow(img, cmap='gray')
    plt.title(f"CORRECT: {class_names[pred_l]}\nConf: {conf:.1f}%", fontsize=9, color='green', fontweight='bold')
    plt.axis('off')

# Plot 5 Failures
for idx, (img, true_l, pred_l, conf) in enumerate(failure_examples):
    plt.subplot(2, 5, idx + 6)
    plt.imshow(img, cmap='gray')
    plt.title(f"MISCLASSIFIED\nTrue: {class_names[true_l]}\nPred: {class_names[pred_l]} ({conf:.1f}%)", fontsize=8, color='red', fontweight='bold')
    plt.axis('off')

plt.suptitle("Qualitative Predictions: High-Confidence Correct Cases (Top) vs. Systematic Failure Modes (Bottom)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 11: Automated Sanity & Integrity Audits

To guarantee complete reproducibility and eliminate deep learning shortcuts or data leakage, we execute the automated 4-part audit suite directly in the notebook.


In [ ]:
from audit_sanity import (
    audit_data_leakage,
    audit_model_integrity,
    audit_eval_mode_and_regularization,
    audit_checkpoint_restoration
)

print("RUNNING AUTOMATED REPRODUCIBILITY SUITE...")
audit_data_leakage()
audit_model_integrity()
audit_eval_mode_and_regularization()
audit_checkpoint_restoration()
print("\n-> ALL SANITY AND INTEGRITY AUDITS PASSED WITH ZERO VIOLATIONS!")


---
## Section 12: Experiment Observability & Bias-Variance Diagnostics (TensorBoard)

To align with industry-standard MLOps practices, our pipeline integrates **TensorBoard** (`torch.utils.tensorboard`) for real-time training telemetry, loss trajectory monitoring, and empirical **Bias vs. Variance decomposition** across all 5 architectures:

- **High Variance Diagnosis (`Diagnosis/Generalization_Gap`)**: Monitored via $\Delta = \mathcal{L}_{\text{val}} - \mathcal{L}_{\text{train}}$. A diverging generalization gap signals overfitting. Regularization (Dropout $p=0.3$, weight decay $10^{-4}$) maintains $\Delta < 0.05$ across all deep networks.
- **High Bias Diagnosis (`Diagnosis/Train_Loss_Floor`)**: Monitored via asymptotic training loss. The Linear model plateaus at $\mathcal{L} \approx 0.42$ (high bias / underfitting), whereas the Custom CNN reaches $\mathcal{L} \approx 0.23$ (low bias / high representational capacity).
- **Telemetry Suite**: Tracks `Loss/Train`, `Loss/Val`, `Accuracy/`, `Macro_F1/`, and `Optimization/Learning_Rate` (Cosine Annealing schedule).


In [ ]:
# 1. Ensure TensorBoard event files are compiled for all 5 models
from export_to_tensorboard import export_histories_to_tensorboard
export_histories_to_tensorboard(log_dir="./runs", history_dir="./outputs/checkpoints")

# 2. Launch interactive TensorBoard dashboard directly inside the notebook (Colab / Jupyter)
%load_ext tensorboard
%tensorboard --logdir runs --port 6006


---
## Section 13: Academic Integrity & AI Usage Disclosure (Handbook Section 5)

Per Handbook Section 5, all generative AI tool interactions for Assignment 1 are documented with all 9 mandatory fields:

| Field | Disclosure Record |
| :--- | :--- |
| **Tool Name & Version** | Claude 3.7 Sonnet (Anthropic), Gemini 2.5 Flash (Google), GitHub Copilot (v1.84) |
| **Member Who Used It** | **Hoàng Trọng Huy Minh** (Student ID: **2452743**) — Sole Contributor (100%) |
| **Development Stage** | Architecture design, data pipeline audit, LaTeX report writing, and sanity testing |
| **Purpose** | Drafting XeLaTeX layout, cross-checking inductive bias formulations, building audit test suite |
| **Affected Sections** | `engine.py`, `models.py`, `audit_sanity.py`, `Goat_A1_Report.tex`, `assignment-01/index.html` |
| **Representative Prompt** | *"Design a custom 3-stage CNN in PyTorch for Fashion-MNIST and verify parameter derivations."* |
| **Editing & Verification** | All code verified through local execution (`pytest`, `audit_sanity.py`) and exact metric validation. |
| **Responsible Member** | **Hoàng Trọng Huy Minh** (100% responsibility for submitted content) |
| **Verification Sources** | PyTorch Documentation, He et al. (2016), Dosovitskiy et al. (2020), Vaswani et al. (2017). |

---
**End of Assignment 01 Reproducibility Walkthrough.**
